## 20_create_source_jobs  (OLD workspace)

Creates the 5 practice jobs via the Jobs REST API 2.1 (migration source):
  1. practice-job-1-daily-notebook   — single task, daily schedule
  2. practice-job-2-multitask        — 4 chained tasks (a -> b -> c -> model)
  3. practice-job-3-parameters       — notebook parameters (base_parameters)
  4. practice-job-4-dependent        — run_job task (triggers 4b target)
  5. practice-job-5-weekly-sql        — SQL logic task, weekly schedule
  (+ internal practice-job-4b-target, the job that 4-dependent triggers;
   it is exported too so the run_job link can be remapped after migration)
Schedules are created PAUSED by default (zero-cost safety).
OLD workspace = Databricks Free Edition: jobs omit compute -> they run
serverless on the free tier (no cluster needed).
Classic fallback workspaces (Community Edition / DBR 12.2): the notebook
attaches your existing cluster automatically.
PREREQ: task notebooks imported at /Users/<you>/migration_practice/tasks/
        and lib/config_lib imported at /Users/<you>/migration_practice/lib/


In [ ]:



dbutils.widgets.text("token", "", "Personal access token (old workspace)")
dbutils.widgets.dropdown("unpause_schedules", "false", ["false", "true"],
                         "Create schedules UNPAUSED (keep 'false' for zero cost)")

TOKEN = dbutils.widgets.get("token").strip()
PAUSE_STATUS = "UNPAUSED" if dbutils.widgets.get("unpause_schedules") == "true" else "PAUSED"
assert TOKEN, "Paste a personal access token into the 'token' widget (Settings -> Developer -> Access tokens)"

# Environment parameters injected into EVERY notebook task. No CATALOG on
# the old workspace: tasks use schema.table only. CONFIG_PATH='' uses the
# built-in dev/stg/prd defaults in lib/config_lib.py.
ENV_PARAMS = {"SCHEMA": "practice", "PARTITION": "all", "CONFIG_PATH": ""}


In [ ]:



import requests

try:
    HOST = dbutils.notebook.getContext().apiUrl().get().rstrip("/")
except Exception:
    dbutils.widgets.text("host", "", "Workspace URL, e.g. https://<your-free-edition-workspace>.cloud.databricks.com")
    HOST = dbutils.widgets.get("host").strip().rstrip("/")
assert HOST.startswith("https://"), f"Bad workspace URL: {HOST!r}"

USER = dbutils.notebook.getContext().userName().get()
TASKS = f"/Users/{USER}/migration_practice/tasks"

S = requests.Session()
S.headers.update({"Authorization": f"Bearer {TOKEN}"})

def api(method, path, body=None):
    r = S.request(method, f"{HOST}{path}", json=body if body is not None else None, timeout=60)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text[:500]}")
    return r.json()

print(f"Workspace : {HOST}")
print(f"User      : {USER}")
print(f"Task path : {TASKS}")


In [ ]:



# Compute resolution: attach existing cluster on classic workspaces,
# omit compute entirely on Free Edition (serverless runs the job for free).
def resolve_existing_cluster_id():
    try:
        clusters = api("GET", "/api/2.0/clusters/list").get("clusters", [])
    except Exception:
        return None
    running = [c for c in clusters if c.get("state") == "RUNNING"]
    usable = running or [c for c in clusters if c.get("state") in ("TERMINATED", "RESIZING")]
    return usable[0]["cluster_id"] if usable else None

CLUSTER_ID = resolve_existing_cluster_id()
if CLUSTER_ID:
    print(f"Existing cluster found (classic workspace): {CLUSTER_ID}")
else:
    print("No cluster required — jobs will run serverless (Free Edition).")

def task(key, notebook, depends_on=None, base_parameters=None):
    t = {"task_key": key, "depends_on": [{"task_key": d} for d in (depends_on or [])],
         "notebook_task": {"notebook_path": f"{TASKS}/{notebook}",
                           "base_parameters": {**ENV_PARAMS, **(base_parameters or {})}}}
    if CLUSTER_ID:
        t["existing_cluster_id"] = CLUSTER_ID
    return t

def job(name, tasks, schedule=None):
    settings = {"name": name, "tags": {"migration_practice": "true"}, "tasks": tasks}
    if schedule:
        settings["schedule"] = schedule
    return settings

DAILY   = {"quartz_cron_expression": "0 0 2 * * ?", "timezone_id": "UTC", "pause_status": PAUSE_STATUS}
WEEKLY  = {"quartz_cron_expression": "0 0 3 ? * SUN", "timezone_id": "UTC", "pause_status": PAUSE_STATUS}


In [ ]:



job_defs = [
    ("practice-job-1-daily-notebook",
     job("practice-job-1-daily-notebook",
         [task("task_a", "10_task_a")], DAILY)),

    ("practice-job-2-multitask",
     job("practice-job-2-multitask",
         [task("task_a", "10_task_a"),
          task("task_b", "11_task_b", depends_on=["task_a"]),
          task("task_c", "12_task_c", depends_on=["task_b"]),
          task("model_task", "15_model_task", depends_on=["task_c"])])),

    ("practice-job-3-parameters",
     job("practice-job-3-parameters",
         [task("param_task", "13_param_task",
               base_parameters={"environment": "practice", "sample_rows": "20"})])),
]

created = {}
for name, settings in job_defs:
    try:
        result = api("POST", "/api/2.1/jobs/create", settings)
        created[name] = result["job_id"]
        print(f"CREATED  {name}  job_id={result['job_id']}")
    except Exception as e:
        print(f"FAILED   {name}: {e}")


In [ ]:



# Dependent job pair: target first, then trigger that runs it.
try:
    target = api("POST", "/api/2.1/jobs/create",
        job("practice-job-4b-target", [task("task_c", "12_task_c")]))
    created["practice-job-4b-target"] = target["job_id"]
    print(f"CREATED  practice-job-4b-target  job_id={target['job_id']}")

    trigger = api("POST", "/api/2.1/jobs/create",
        job("practice-job-4-dependent",
            [{"task_key": "run_target", "run_job_task": {"job_id": target["job_id"]}}]))
    created["practice-job-4-dependent"] = trigger["job_id"]
    print(f"CREATED  practice-job-4-dependent  job_id={trigger['job_id']}")
except Exception as e:
    print(f"FAILED   dependent job pair: {e}")


In [ ]:



try:
    weekly = api("POST", "/api/2.1/jobs/create",
        job("practice-job-5-weekly-sql", [task("sql_task", "14_sql_task")], WEEKLY))
    created["practice-job-5-weekly-sql"] = weekly["job_id"]
    print(f"CREATED  practice-job-5-weekly-sql  job_id={weekly['job_id']}")
except Exception as e:
    print(f"FAILED   practice-job-5-weekly-sql: {e}")


In [ ]:



print("\n=== SOURCE JOBS READY ===")
for name, jid in created.items():
    print(f"  {name}  ->  {HOST}/jobs/{jid}")
print("\nAll schedules are PAUSED (zero cost). Validate one run per job later in the NEW workspace.")
